# Imports & Setup

In [1]:
pip uninstall -y datasets > /dev/null 2>&1 && pip install datasets==3.6.0 > /dev/null 2>&1 && pip install evaluate > /dev/null 2>&1 && pip install rouge_score > /dev/null 2>&1

In [2]:
import math
import os
import random
import pandas as pd
import numpy as np
import torch
import evaluate
from google.colab import drive

from transformers import (
  T5TokenizerFast as T5Tokenizer,
  T5ForConditionalGeneration,
  Seq2SeqTrainingArguments,
  Seq2SeqTrainer,
  DataCollatorForSeq2Seq
)
from datasets import load_dataset, concatenate_datasets, Dataset, DatasetDict
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
# Mount Google Drive
drive.mount('/content/drive')

# Define Drive Path
DRIVE_PATH = "/content/drive/MyDrive/Project/Text Summarizer/ModelT5Base"

# Create directory if it doesn't exist
if not os.path.exists(DRIVE_PATH):
    os.makedirs(DRIVE_PATH)
    print(f"Created directory: {DRIVE_PATH}")
else:
    print(f"Directory exists: {DRIVE_PATH}")

Mounted at /content/drive
Directory exists: /content/drive/MyDrive/Project/Text Summarizer/ModelT5Base


# Dataset Preparation & Information

In [4]:
TRAIN_SAMPLES = 5000 # Increased slightly for better style learning

print("Loading datasets...")
xsum = load_dataset('xsum', trust_remote_code=True, split='train')
cnn_dailymail = load_dataset('cnn_dailymail', '3.0.0', split='train')

xsum = xsum.select(range(TRAIN_SAMPLES))
cnn_dailymail = cnn_dailymail.select(range(TRAIN_SAMPLES))

xsum = xsum.remove_columns(['id'])
xsum = xsum.rename_columns({
  'document' : 'text',
  'summary' : 'summary'
})

cnn_dailymail = cnn_dailymail.remove_columns(['id'])
cnn_dailymail = cnn_dailymail.rename_columns({
  'article' : 'text',
  'highlights' : 'summary'
})

dataset = concatenate_datasets([xsum, cnn_dailymail])
full_dataset = dataset.train_test_split(test_size=0.1, shuffle=True)

dataset_train = full_dataset['train']
dataset_valid = full_dataset['test'].select(range(200))

print(f"Training samples: {len(dataset_train)}")
print(f"Validation samples: {len(dataset_valid)}")

Loading datasets...


README.md: 0.00B [00:00, ?B/s]

xsum.py: 0.00B [00:00, ?B/s]

data/XSUM-EMNLP18-Summary-Data-Original.(…):   0%|          | 0.00/255M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/204045 [00:00<?, ? examples/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Training samples: 9000
Validation samples: 200


# Configurations & Parameters

In [5]:
MODEL_NAME = 'google/flan-t5-base'
BATCH_SIZE = 4
EPOCHS = 3
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 150
LEARNING_RATE = 3e-4
SEED = 42
GRAD_ACCUMULATION = 2

HARSH_THRESHOLD = 0.30
STANDARD_THRESHOLD = 0.60

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Tokenization & Style Processing

In [6]:
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
rouge = evaluate.load("rouge")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [7]:
def assign_length_category(text, summary):
    text_len = len(text.split())
    summary_len = len(summary.split())
    if text_len == 0:
        ratio = 0
    else:
        ratio = summary_len / text_len

    if ratio <= HARSH_THRESHOLD:
        return "harsh"
    elif ratio <= STANDARD_THRESHOLD:
        return "standard"
    else:
        return "detailed"

def preprocess_function(examples):
    # 1. Determine Style and Format Input
    inputs = []
    for text, summary in zip(examples["text"], examples["summary"]):
        style = assign_length_category(text, summary)
        # Format: "summarize {style}: {text}"
        inputs.append(f"summarize {style}: {text}")

    # 2. Tokenize Inputs
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length",
    )

    # 3. Tokenize Targets
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing and applying style labels...")
tokenized_train = dataset_train.map(preprocess_function, batched=True)
tokenized_valid = dataset_valid.map(preprocess_function, batched=True)

Tokenizing and applying style labels...


Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

# Model Loading & Training

In [8]:
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.gradient_checkpointing_enable()
model.to(device)

import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.clip(predictions, 0, tokenizer.vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    return {k: round(v, 4) for k, v in result.items()}

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [11]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    # --- Checkpoints & Output ---
    output_dir=DRIVE_PATH,
    load_best_model_at_end=True,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    report_to="none",

    # --- Core Training Hyperparameters ---
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    label_smoothing_factor=0.1,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,

    # --- Evaluation ---
    eval_strategy="steps",
    eval_steps=500,
    per_device_eval_batch_size=BATCH_SIZE,

    # --- Performance Optimization ---
    fp16=True,
    gradient_checkpointing=True,

    # --- Generation (The Accuracy Boosters) ---
    predict_with_generate=True,
    generation_max_length=150,
    generation_num_beams=4,
)

# 3. INITIALIZE TRAINER
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# 4. RUN TRAINING
torch.cuda.empty_cache()
print("Starting training...")
trainer.train()

print(f"Saving final model to {DRIVE_PATH}...")
trainer.save_model(DRIVE_PATH)
tokenizer.save_pretrained(DRIVE_PATH)

Starting training...


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
500,0.000000,nan,0.249500,0.090200,0.196700,0.196400
1000,0.000000,nan,0.249500,0.090200,0.196700,0.196400
1500,0.000000,nan,0.249500,0.090200,0.196700,0.196400
2000,0.000000,nan,0.249500,0.090200,0.196700,0.196400
2500,0.000000,nan,0.249500,0.090200,0.196700,0.196400
3000,0.000000,nan,0.249500,0.090200,0.196700,0.196400


Saving final model to /content/drive/MyDrive/Project/Text Summarizer/ModelT5Base...


('/content/drive/MyDrive/Project/Text Summarizer/ModelT5Base/tokenizer_config.json',
 '/content/drive/MyDrive/Project/Text Summarizer/ModelT5Base/special_tokens_map.json',
 '/content/drive/MyDrive/Project/Text Summarizer/ModelT5Base/spiece.model',
 '/content/drive/MyDrive/Project/Text Summarizer/ModelT5Base/added_tokens.json',
 '/content/drive/MyDrive/Project/Text Summarizer/ModelT5Base/tokenizer.json')

# Model Testing & Evaluation (Inference)

In [12]:
# Load model from Drive for Inference testing
print(f"Loading model from {DRIVE_PATH}...")
model = T5ForConditionalGeneration.from_pretrained(DRIVE_PATH).to(device)
tokenizer = T5Tokenizer.from_pretrained(DRIVE_PATH)

Loading model from /content/drive/MyDrive/Project/Text Summarizer/ModelT5Base...


In [13]:
def generate_summary_dynamic(text, style, model, tokenizer):
    """
    Generates a summary based on the requested style using the dynamic length logic
    from inference.ipynb.
    """
    model.eval()
    input_text = f"summarize {style}: {text}"

    # Calculate input word count for dynamic targets
    input_word_count = len(text.split())

    inputs = tokenizer(input_text, max_length=1024, truncation=True, return_tensors="pt").to(device)

    # DYNAMIC LENGTH LOGIC (From inference.ipynb)
    if style == "harsh":
        # Target: ~30% of original
        min_len = int(input_word_count * 0.25)
        max_len = int(input_word_count * 0.35)
        length_penalty = 2.0 # Force brevity
    elif style == "standard":
        # Target: ~50-60% of original
        min_len = int(input_word_count * 0.45)
        max_len = int(input_word_count * 0.65)
        length_penalty = 1.0
    else: # detailed
        # Target: ~80% of original
        min_len = int(input_word_count * 0.70)
        max_len = int(input_word_count * 0.90)
        length_penalty = 0.5 # Encourage length

    # Safety check
    if min_len >= max_len:
        min_len = max(5, max_len - 10)

    # Hard cap for max_len to prevent errors if text is huge
    max_len = min(max_len, 512)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_len,
            min_length=min_len,
            num_beams=4,
            length_penalty=length_penalty,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [14]:
text = """
    Russian President Vladimir Putin said that a US plan to end the war in Ukraine could “form the basis for future agreements”
    but renewed threats to seize more territory by force unless Kyiv withdraws. Speaking to reporters in Bishtek, in the Central
    Asian republic of Kyrgyzstan on Thursday, Putin confirmed the Kremlin was expecting a US delegation headed by Special Envoy
    Steve Witkoff to visit Moscow early next week, adding that the Kremlin was ready for “serious discussion”. But chances of a
    swift breakthrough appear slim after Putin repeated his maximalist demands, saying the war in Ukraine will only end “once
    Ukrainian troops withdraw from the territories they occupy.” “If they don’t withdraw, we will achieve this through military
    means,” the Russian leader said. Russia is occupying some 20% of the territory recognized under international law as part of
    sovereign Ukraine, including almost all of the Luhansk region, and parts of the Donetsk, Kherson and Zaporizhzhia regions.
    Moscow is demanding Ukraine surrender the entirety of these four regions, which it has annexed but not fully conquered. Russia
    has made some gains along the eastern Ukrainian frontline in recent weeks, most significantly around the city of Pokrovsk. Still,
    the Institute for the Study of War, a US-based conflict monitor, said on Thursday that data on Russian forces’ rate of advance
    indicates that “a Russian military victory in Ukraine is not inevitable, and a rapid Russian seizure of the rest of Donetsk Oblast
    is not imminent.” Crucially, the area Russia is demanding includes the “fortress belt” of heavily defended towns and cities seen
    that are seen as vital for Ukrainian security. Kyiv and its European allies have made it clear that territorial concessions are a red line
    for them. Putin’s remarks on Thursday were the strongest indication that Russia is not willing to move after US officials, including
    Trump himself, touted “tremendous progress” being made on their efforts to end the war. This came after Ukrainian and European
    officials strongly opposed and then revised the 28-point peace plan which was drafted by the US with apparent strong input from Russia.
    The original plan reflected Russia’s extensive wishlist, and included a demand for Ukraine to reduce its army and be barred from
    joining NATO. Putin said on Thursday that he was expecting Witkoff to arrive to Moscow early next week, presumably to discuss the
    new draft of the plan, the exact wording of which is not yet known. Putin said he has been informed of the latest discussions and that
    the plan could “form the basis for future agreements.” “It would be impolite of me to speak of final agreements now,” Putin added.
"""

print(f"Length of Original Text: {len(text.split())}")
print()

for style in ["harsh", "standard", "detailed"]:
    summary = generate_summary_dynamic(text, style, model, tokenizer)
    print(f"\n--- {style.capitalize()} Summary ({len(summary.split())} words) ---")
    print(summary)

Length of Original Text: 440


--- Harsh Summary (83 words) ---
Russian President Vladimir Putin says war in Ukraine will only end ‘once Ukrainian troops withdraw from territories they occupy’ but renewed threats to seize more territory by force unless Kyiv withdraws unless it withdraws from the territories it occupys – a move that is seen as a ‘red line’ for Ukraine’s allies – says he expects a US delegation to arrive early next week to discuss the new draft of a peace plan drafted by the US with apparent strong input from Russia.

--- Standard Summary (135 words) ---
Russian President Vladimir Putin says war in Ukraine will only end ‘once Ukrainian troops withdraw from territories they occupy’ but renewed threats to seize more territory by force unless Kyiv withdraws unless it withdraws from the territories it occupys – a claim he says is ‘impossible’ if Ukraine refuses to withdraw from the territory it occupies. The Russian leader said he was expecting a US delegation headed by Spec

In [15]:
from google.colab import runtime
runtime.unassign()